[Guide Index](README.md)

# 04.02 SRL: how a rule set runs

[04.01](04.01-srl-sparql-rules.ipynb) shows how to write and run rules. This guide explains:
- the order a rule set's rules run in (the plan),
- the rules that run once,
- the rule sets that are rejected and why,
- what running the rules again adds,
- asking questions of a rule set, and keeping the derived triples,
- `DATA` blocks and imports.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import NamedNode, StarLayerGraph
from starlayer.sparql import srl

EX = "http://example.org/"


def ex(name):
    """A term in the example namespace, as a pyoxigraph NamedNode."""
    return NamedNode(EX + name)


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses.
    (A new StarLayerGraph has no prefixes; the ones used here are bound explicitly.)"""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 04.02.1 The base graph

The examples use a small software-dependency graph, the same running example as [the SRL spec's worked example](https://www.w3.org/TR/sparql12-rl/#worked-example): a `:frontend` depends on an `:app`, which depends on a `:db` that has a known vulnerability. A `:logger` component has no vulnerabilities at all. The rules below spread the vulnerability along the dependency chain.

In [2]:
base = StarLayerGraph(prefixes={"ex": EX})
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .  # no vulnerability - R4 will mark it safe

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")


print("data graph")
show(base)

data graph
@prefix ex: <http://example.org/> .

ex:app a ex:Component ;
    ex:dependsOn ex:db .

ex:db a ex:Component ;
    ex:hasVulnerability ex:vuln1 .

ex:frontend a ex:Component ;
    ex:dependsOn ex:app .

ex:logger a ex:Component .

ex:vuln1 ex:severity 9.1 .



## 04.02.2 The rules

Four rules work on this graph. R1 and R2 find which components are exposed to a vulnerability: R1 for a vulnerability a component has itself, and R2, which is recursive, for one it reaches through `ex:dependsOn`. R3 marks a component critically exposed if a vulnerability has `severity >= 9.0` (`FILTER`). R4 marks a component safe if it is *not* critically exposed (`NOT { }`).

R4 needs the result of R3, and R3 needs the result of R1 and R2. The next section shows how SRL works out an order for them.

In [3]:
rules_text = """
PREFIX ex: <http://example.org/>

# R1: a component is exposed to any vulnerability it has directly.
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }

# R2: exposure propagates along dependsOn (recursive - depends on itself).
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

# R3: severity >= 9.0 counts as a critical vulnerability.
RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

# R4: not critically exposed -> safe to deploy.
RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = srl.parse_ruleset(rules_text)
inferred = ruleset.infer(base)
inferred.bind("ex", EX)
show(inferred)

@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:db ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:logger ex:status ex:safeToDeploy .



## 04.02.3 The plan

Before running anything, SRL works out an order for the rules. `ruleset.plan()` shows the plan:
- the **strata** (stages) in the order they run, and
- the **dependency edges** between rules that the strata were worked out from.

Rules are identified by their position in `ruleset.rules`, so R1 is index 0.

Each stratum finishes before the next starts. Within a stratum the ordinary rules repeat until nothing new appears.

An edge `source -> target` means the `source` rule's body reads something the `target` rule's head can produce. It is **open** for an ordinary dependency: the two can sit in the same stratum and repeat together. It is **closed** when the source must see everything the target produces before it runs: for example when it reads the target through `NOT { }`, or when the source is a run-once rule (next section). A closed edge forces the source into a *later* stratum than the target.

For the four rules above, R4 has a closed edge to R3, so R4 is in stratum 1 and R1, R2 and R3 are in stratum 0:

In [4]:
plan = ruleset.plan()


for stratum in plan.strata:
    print(f"stratum {stratum.number}:", ", ".join(f"R{r.index + 1}" for r in stratum.general))


for edge in sorted(plan.edges, key=lambda e: (e.source, e.target)):
    print(f"R{edge.source + 1} -> R{edge.target + 1} ({edge.label})")

assert [[r.index for r in s.general] for s in plan.strata] == [[0, 1, 2], [3]]
assert {(e.source, e.target): e.label for e in plan.edges}[(3, 2)] == "closed"

stratum 0: R1, R2, R3
stratum 1: R4
R2 -> R1 (open)
R2 -> R2 (open)
R3 -> R1 (open)
R3 -> R2 (open)
R4 -> R3 (closed)


### Rule order doesn't matter

The plan comes from the rules' *structure*, not from the order they are written in. Below, R4 is written and listed before R3, and each rule is given an id so the plan can name it. R4 still lands in the later stratum, and `infer()` produces exactly the same result.

In [5]:
reordered_rules_text = """
PREFIX ex: <http://example.org/>

RULE ex:R1 { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE ex:R2 { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

# R4 listed before R3 on purpose - it has a closed dependency on R3 through NOT { }.
RULE ex:R4 { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }

RULE ex:R3 { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }
"""

reordered_ruleset = srl.parse_ruleset(reordered_rules_text)

for stratum in reordered_ruleset.plan().strata:
    print(f"stratum {stratum.number}:", ", ".join(base.qname(r.id) for r in stratum.general))

reordered_inferred = reordered_ruleset.infer(base)
print("same result regardless of rule order:", set(reordered_inferred) == set(inferred))
assert set(reordered_inferred) == set(inferred)

stratum 0: ex:R1, ex:R2, ex:R3
stratum 1: ex:R4
same result regardless of rule order: True


## 04.02.4 Run-once rules

Some rules run exactly once, at their place in the plan, instead of repeating. A rule is a **run-once rule** when:
- its body has a `SET` or a group element, or
- its head has a blank node, or a triple term that contains a variable.

Rules containing these elements can create new terms that were not in the graph before. A rule that only reuses terms already in the graph can only ever produce a finite number of triples, so the repeated passes must stop at some point. A rule that creates a new term could go on creating new terms for ever. The SRL specification requires that such a rule runs only once.

A run-once rule waits for every rule it relies on to finish, so all its dependencies are closed. It is placed in the first stratum after those of all the rules it relies on, and the rules that rely on it are placed after it. Within a stratum the run-once rules run first, one time each, and then the ordinary rules repeat until no new triples are created.

Below, `score` is a run-once rule: it computes a risk score from the exposure rules, so it runs after them. `high` is an ordinary rule that reads the score, and `score2` is a second run-once rule that reads `score`:

In [6]:
staged_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE ex:exposure1 { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v }
RULE ex:exposure2 { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v }
RULE ex:score     { ?x ex:score ?r }     WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . SET(?r := ?s * 10) }
RULE ex:high      { ?x ex:high true }    WHERE { ?x ex:score ?r . FILTER(?r > 50) }
RULE ex:score2    { ?x ex:score2 ?r }    WHERE { ?x ex:score ?s . SET(?r := ?s + 1) }
RULE ex:unrelated { ?x ex:q ?y }         WHERE { ?x ex:p ?y }
""")

for stratum in staged_rules.plan().strata:
    once = ", ".join(base.qname(r.id) for r in stratum.run_once) or "-"
    repeat = ", ".join(base.qname(r.id) for r in stratum.general) or "-"
    print(f"stratum {stratum.number}: run once: {once:12} | repeat: {repeat}")

stratum 0: run once: -            | repeat: ex:exposure1, ex:exposure2, ex:unrelated
stratum 1: run once: ex:score     | repeat: ex:high
stratum 2: run once: ex:score2    | repeat: -


`score` and `high` share a stratum: `score` runs once first, and then `high` repeats. `score2` is a run-once rule that reads what `score` makes, so it has a stratum of its own after `score`. Run-once rules can depend on each other. What they cannot do is depend on each other in a loop, as the next section shows.

## 04.02.5 Cycles and rejected rule sets

Not every rule set can be run. `plan()` and `infer()` check that the rules can be placed in a solvable order (the plan from section 04.02.3), and `validate_sparql_tree()` reports it (see [06.02.4](06.02-sparql-and-srl-as-rdf.ipynb)). They reject a rule set that has no valid order by raising `ExecutionError` (kind `"rule"`). The cases below show what is accepted and what is rejected.

The check starts from which rules **depend on** which. A rule depends on another when a pattern in its body could match a triple the other rule's head creates. The check compares the rules' patterns, not the data, so a rule that *could* match the output of another rule counts as a dependency, even if the data would never make it do so. A **cycle** is a loop of such dependencies, including a rule that depends on itself.

Cycles are normal for ordinary rules. R2 depends on itself, and two ordinary rules can depend on each other: they repeat together until nothing new appears.

A cycle is rejected when it goes through a `NOT { }`, a group element or a run-once rule. Such a rule has to wait for the rules it reads from to finish, and on the cycle one of them is itself. It would be waiting for itself, so no possible order exists. A rule like this that is not on a cycle is fine: R4 reads R3 through `NOT { }`, but R3 does not depend on R4. The specification leaves the outcome for these rule sets undefined, so StarLayer rejects them with `ExecutionError` (kind `"rule"`).

Each case below is a rule set parsed and planned:

In [7]:
from starlayer.errors import ExecutionError, reraise_unless

PREFIXES = "PREFIX ex: <http://example.org/>\n"

cases = [
    ("a rule that depends on itself",
     "RULE { ?x ex:r ?z } WHERE { ?x ex:p ?y . ?y ex:r ?z } RULE { ?x ex:r ?y } WHERE { ?x ex:p ?y }"),
    ("two ordinary rules that depend on each other",
     "RULE { ?x ex:even ?y } WHERE { ?x ex:p ?z . ?z ex:odd ?y } RULE { ?x ex:odd ?y } WHERE { ?x ex:p ?y } "
     "RULE { ?x ex:odd ?y } WHERE { ?x ex:p ?z . ?z ex:even ?y }"),
    ("a chain of run-once rules (no cycle)",
     "RULE { ?x ex:b ?n } WHERE { ?x ex:a ?m . SET(?n := ?m + 1) } RULE { ?x ex:c ?n } WHERE { ?x ex:b ?m . SET(?n := ?m + 1) }"),
    ("a rule that negates its own output (NOT)",
     "RULE { ?x ex:flagged ex:yes } WHERE { ?x a ex:Component . NOT { ?x ex:flagged ex:yes } }"),
    ("a SET rule that feeds itself",
     "RULE { ?x ex:level ?n } WHERE { ?x ex:level ?m . SET(?n := ?m + 1) }"),
    ("two run-once rules that feed each other",
     "RULE { ?x ex:b ?n } WHERE { ?x ex:a ?m . SET(?n := ?m + 1) } RULE { ?x ex:a ?n } WHERE { ?x ex:b ?m . SET(?n := ?m + 1) }"),
    ("a blank node feeding itself",
     "RULE { ?x ex:child _:c } WHERE { ?x ex:child ?y }"),
]

for label, text in cases:
    try:
        srl.parse_ruleset(PREFIXES + text).plan()
        print(f"{label:48} accepted")
    except ExecutionError as error:
        reraise_unless(error, "rule")
        print(f"{label:48} ExecutionError (kind rule)")

a rule that depends on itself                    accepted
two ordinary rules that depend on each other     accepted
a chain of run-once rules (no cycle)             accepted
a rule that negates its own output (NOT)         ExecutionError (kind rule)
a SET rule that feeds itself                     ExecutionError (kind rule)
two run-once rules that feed each other          ExecutionError (kind rule)
a blank node feeding itself                      ExecutionError (kind rule)


## 04.02.6 Validation errors

A rule set can be wrong before it runs. `parse_ruleset()` loads any text that is SRL, even if the rule set has problems, so it can be inspected and fixed. The problems are reported when the rule set is run or checked:

| What is wrong | Error | When |
|---|---|---|
| the text is not valid SRL, or it uses a property path other than `/` and `^` | `FormatError` (kind `"srl"`) | `srl.parse_ruleset()` |
| a variable in a rule's head is not bound by its body, or a variable name is reused in another part of the rule | `FormatError` (kind `"srl"`) | `infer()`, `query()` and `materialize()`; `validate_sparql_tree()` reports it as a violation |
| the rule set does not conform to StarLayer's SHACL shapes for SRL | a violation in the report | `validate_sparql_tree()` |
| a cycle goes through a `NOT { }`, a group element or a run-once rule (section 04.02.5) | `ExecutionError` (kind `"rule"`) | `plan()` and `infer()`; `validate_sparql_tree()` reports it as a violation |
| an import cannot be read (section 04.02.11) | `ExecutionError` (kind `"import"`) | `srl.parse_ruleset(imports_loader=)` and `srl.resolve_imports()` |
| an import is not a valid rule set (section 04.02.11) | `FormatError` (kind `"srl"`) | `srl.parse_ruleset(imports_loader=)` and `srl.resolve_imports()` |
| a rule set that still has `IMPORTS` is run (section 04.02.11) | `ExecutionError` (kind `"import"`) | `infer()` and `query()` |

Every variable used in a rule's head must be bound by its body ([SRL section 4.2](https://www.w3.org/TR/sparql12-rl/#well-formedness-conditions)). A variable name may not be reused either: a name that appears only inside a `NOT` or a group pattern cannot appear in another part of the rule, and a `SET` or aggregation variable must be new (a variable bound before a `NOT` can be used inside it). A rule set that breaks one of these parses, and raises `FormatError` when it is run, so a typo in a variable name does not give a wrong answer later. `validate_sparql_tree()` reports the same problems for each element of the rule set before it is run (see section 06.02.4 of the [SPARQL queries, updates and rules as RDF guide](06.02-sparql-and-srl-as-rdf.ipynb)).

In [8]:
from starlayer.errors import FormatError

broken = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:flaggedBy ?reviewer } WHERE { ?x ex:hasVulnerability ?v . }
""")
try:
    broken.infer(StarLayerGraph())
except FormatError as e:
    print("FormatError:", e)


FormatError: SRL: rule set is not well-formed per §4.2: unbound_head_variable (?reviewer in rule head)


## 04.02.7 Running a rule set again

A rule set derives everything it can in one call to `infer()`. If the same rules are run over the base graph together with what they inferred, ordinary rules add nothing new, because the result is already complete. Rules with a `SET` or a group element add nothing either, since they depend only on rules that have finished.

Two kinds of rule can add triples on a second run:
- a rule with a **blank node** in its head, which creates fresh nodes on every run;
- a rule that uses **`WHERE GROUND`** (section 04.02.10), which sees the earlier results as base data on the second run.

Whether to run again, and how many times, is up to the application. The function below counts the triples a second run adds for each kind of rule:

In [9]:
EXPOSURE = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v }
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v }
"""


def second_run(rules_text):
    """The number of triples a rule set derives, and how many a second run over the base graph plus those adds."""
    rules = srl.parse_ruleset(rules_text)
    first = rules.infer(base)
    combined = base.copy()
    combined += first
    again = rules.infer(combined)
    return len(first), sum(1 for triple in again.triples() if triple not in combined)


examples = {
    "ordinary rules": EXPOSURE,
    "a SET rule": EXPOSURE + "RULE { ?x ex:score ?r } WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . SET(?r := ?s * 10) }",
    "a blank node in a head": EXPOSURE + "RULE { ?x ex:ticket _:t } WHERE { ?x ex:exposedTo ?v }",
    "WHERE GROUND": "PREFIX ex: <http://example.org/> "
                  "RULE { ?x ex:flagged ex:yes } WHERE GROUND { ?x ex:hasVulnerability ?v } "
                  "RULE { ?y ex:seenAsFlagged ex:yes } WHERE GROUND { ?y ex:flagged ex:yes }",
}

for label, text in examples.items():
    derived, added = second_run(text)
    print(f"{label:24} first run: {derived} triples | a second run adds: {added}")

ordinary rules           first run: 3 triples | a second run adds: 0
a SET rule               first run: 6 triples | a second run adds: 0
a blank node in a head   first run: 6 triples | a second run adds: 3
WHERE GROUND             first run: 1 triples | a second run adds: 1


## 04.02.8 Asking questions of a rule set

`infer()` returns everything a rule set derives. To ask one question, use `ruleset.query(base, goal)` as in [04.01](04.01-srl-sparql-rules.ipynb). `query()` runs only the rules the goal depends on: the rules that are needed to derive the goal. The rest of the rule set is not run. Each call to `query()` runs its rules again.

An application that asks many questions of a graph can run the rules once and keep the result with `ruleset.materialize(graph)`. The `Materialization` it returns holds the base graph and the derived triples, and `kept.query(goal)` answers a goal from them without running the rule set again.

Use `materialize()` when many questions are asked of one graph. To merely obtain the derived triples, `infer()` is sufficient: `kept.inferred` is the same graph that `infer()` returns. For one question, `query()` is often faster because it is limited to the rules it needs.

A materialization is a snapshot of the graph when it is created. It does not follow the graph. `kept.stale` is `True` once something has been written through StarLayer to the store the graph is on since the rules ran (a write to another graph of the same store counts, and a write to another store does not), and `kept.refresh()` runs the rules again. `kept.close()` drops the kept triples and the copy of the graph.

In [10]:
# Work on a copy of the graph, so the write below does not change base
working = StarLayerGraph(prefixes={"ex": EX})
working.extend(list(base))

# The goal: which components are critically exposed?
goal = "?x ex:status ex:criticallyExposed"

# Run the rules once and keep the result, a Materialization
kept = ruleset.materialize(working)
print("derived triples in kept:", len(kept.inferred))

# Ask a goal of the kept triples: no rule is run
rows = kept.query(goal)
print("critically exposed:", sorted(working.qname_term(term) for row in rows for term in row.values()))

# A new component that depends on the vulnerable database
working.add((ex("worker"), ex("dependsOn"), ex("db")))
print("stale after a write:", kept.stale)

# The kept result is a snapshot, so it does not have the worker yet
worker_goal = "ex:worker ex:status ex:criticallyExposed"
before = bool(kept.query(worker_goal))
print("worker critically exposed before refresh:", before)

# Run the rules again on the graph as it is now
kept.refresh()
after = bool(kept.query(worker_goal))
print("stale after refresh:", kept.stale)
print("worker critically exposed after refresh:", after)

# Drop the kept triples and the copy of the graph
kept.close()

derived triples in kept: 7
critically exposed: ['ex:app', 'ex:db', 'ex:frontend']
stale after a write: True
worker critically exposed before refresh: False
stale after refresh: False
worker critically exposed after refresh: True


## 04.02.9 `DATA` blocks

A rule set can carry its own ground facts in a `DATA { }` block. The facts travel with the rules and get combined with the base graph before the rules are run.

A rule matches them exactly as it matches triples from the base graph, and they also appear in the output.

The spec names three graphs, and this section depends on telling them apart:
- the **base graph** is the data passed to `infer()`;
- the **evaluation graph** is what the rules match as they run: the base graph, plus the rule set's `DATA { }` facts, plus every triple inferred so far;
- the **inference graph** is what `infer()` returns: the inferred triples, not including anything already in the base graph.

A `DATA { }` block behaves like a rule with an empty body: its triples join the evaluation graph without any rule being run.

Here the rule set records which team owns two of the components, and one rule derives a `reportsTo` triple from each:

In [11]:
# `base` is the software-dependency graph from section 04.02.1
team_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

DATA { ex:db ex:team ex:platform . ex:app ex:team ex:web . }

RULE { ?x ex:reportsTo ?t } WHERE { ?x ex:team ?t }
""")

team_inferred = team_rules.infer(base)
team_inferred.bind("ex", EX)
show(team_inferred)

assert (ex("db"), ex("team"), ex("platform")) in team_inferred      # the DATA facts are in the output
assert (ex("db"), ex("reportsTo"), ex("platform")) in team_inferred # and the rule matched them

@prefix ex: <http://example.org/> .

ex:app ex:reportsTo ex:web ;
    ex:team ex:web .

ex:db ex:reportsTo ex:platform ;
    ex:team ex:platform .



## 04.02.10 `WHERE GROUND` and `NOT GROUND`

A `DATA { }` block holds facts. The keyword `GROUND`, after `WHERE` or after `NOT`, is a different thing: it limits what a rule matches to the base graph. (The draft spelled it `DATA` until November 2026, so an older rule set may say `WHERE DATA` or `NOT DATA`; StarLayer now reads only `GROUND` there and says so in the error.)

### `WHERE GROUND`: matching base data

A rule typically matches against the whole evaluation graph. A triple created in one pass of a rule is available to subsequent passes of rules.  

Using the keyword `GROUND` after `WHERE` changes that behavior and the rule matches only **base data**. Triples the rules have inferred are not available to the rule.

`WHERE GROUND` does **not** mean "match the `DATA { }` block", despite the word. The base graph is the data passed to `infer()`, and it is separate from the rule set's own `DATA { }` facts.

| A rule's `WHERE` matches... | plain `WHERE { }` | `WHERE GROUND { }` |
|---|---|---|
| the base graph passed to `infer()` | yes | yes |
| the rule set's own `DATA { }` facts | yes | **no** |
| triples inferred by the rules so far | yes | **no** |

Use `WHERE GROUND { }` for a rule that should look only at the data supplied in the base graph, never at the rule set's own `DATA` facts or at other rules' conclusions. 

The same `DATA` fact is used below by three rules:
- the first uses a plain `WHERE`, so it matches the `DATA` fact;
- the second uses `WHERE GROUND` for the same pattern, so it matches nothing, because `ex:team` exists only in the `DATA` block, not in the base graph;
- the third uses `WHERE GROUND` for `ex:hasVulnerability`, which *is* in the base graph, so it matches.

In [12]:
# `base` is the software-dependency graph from section 04.02.1
scope_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

DATA { ex:db ex:team ex:platform . }

RULE { ?x ex:seenInData ?t }  WHERE      { ?x ex:team ?t }
RULE { ?x ex:seenInBase ?t }  WHERE GROUND { ?x ex:team ?t }
RULE { ?x ex:flagged ex:yes } WHERE GROUND { ?x ex:hasVulnerability ?v }
""")

scope_inferred = scope_rules.infer(base)
scope_inferred.bind("ex", EX)
show(scope_inferred)

assert (ex("db"), ex("seenInData"), ex("platform")) in scope_inferred       # ordinary WHERE sees the DATA fact
assert not list(scope_inferred.triples((None, ex("seenInBase"), None)))  # WHERE GROUND does not
assert (ex("db"), ex("flagged"), ex("yes")) in scope_inferred               # WHERE GROUND sees the base graph

@prefix ex: <http://example.org/> .

ex:db ex:flagged ex:yes ;
    ex:seenInData ex:platform ;
    ex:team ex:platform .



### `NOT GROUND`: filling in a default

`GROUND` can also be used on a single `NOT`. The spec's own use is a **default value**: derive a value only when the base graph does not already state one. 

The rule below gives every vulnerable component a review priority unless the base graph already has one. The cell builds its own small base graph (just `ex:db`, with a vulnerability) and runs the rule three ways: with no priority in the base graph, with one already there, and with a plain `NOT` instead of `NOT GROUND`.

The rule negates the very predicate it produces (`ex:reviewPriority`). With a plain `NOT { }` the rule cannot be ordered (section 04.02.5), because the rule would need its own output to be finished before it could run. `NOT GROUND { }` looks only at the base graph, so there is no cycle, and the rest of the rule body can still match inferred triples.

In [13]:
# The base graph for this example: ex:db is a component with a known vulnerability.
# (Section 04.02.1's graph is not used here, so the example stands on its own.)
review_base = StarLayerGraph(prefixes={"ex": EX})
review_base.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:db a ex:Component ; ex:hasVulnerability ex:vuln1 .
""", format="turtle")

# The rule: a component with a vulnerability gets a high review priority,
# unless the base graph already states a priority for it.
default_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:reviewPriority ex:high }
WHERE { ?x ex:hasVulnerability ?v . NOT GROUND { ?x ex:reviewPriority ?existing } }
""")

# Case 1: the base graph has no priority for ex:db, so the default is derived.
case1 = default_rules.infer(review_base)
case1.bind("ex", EX)
print("Case 1 - base graph has no priority. Derived:")
show(case1)

# Case 2: the base graph already states a priority for ex:db, so nothing is derived.
review_base_with_priority = StarLayerGraph()
review_base_with_priority += review_base
review_base_with_priority.add((ex("db"), ex("reviewPriority"), ex("low")))
case2 = default_rules.infer(review_base_with_priority)
print("Case 2 - base graph already has ex:db ex:reviewPriority ex:low. Triples derived:", len(case2))

assert (ex("db"), ex("reviewPriority"), ex("high")) in case1
assert len(case2) == 0

# Case 3: the same rule with a plain NOT (matching the whole evaluation graph) cannot be ordered.
try:
    srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?x ex:reviewPriority ex:high }
    WHERE { ?x ex:hasVulnerability ?v . NOT { ?x ex:reviewPriority ?existing } }
    """).plan()
except ExecutionError as e:
    print("Case 3 - the same rule with a plain NOT cannot be ordered:", type(e).__name__, f"(kind {e.kind})")

Case 1 - base graph has no priority. Derived:
@prefix ex: <http://example.org/> .

ex:db ex:reviewPriority ex:high .

Case 2 - base graph already has ex:db ex:reviewPriority ex:low. Triples derived: 0
Case 3 - the same rule with a plain NOT cannot be ordered: ExecutionError (kind rule)


## 04.02.11 Importing rule sets

A rule set can import other rule sets with `IMPORTS <iri>`, so rules can be kept in shared libraries. The imported rules are merged into the rule set. Importing can involve reading a document, which has security implications, so StarLayer never reads a library unless the application says how. There are three steps:

1. **Have the libraries available as SRL text.** They can come from anywhere: files, a URL, a database. Here they are held as strings in the application.
2. **Write a loader.** A loader is a function that takes the IRI of an import and returns that library's text. It is the one place that decides where libraries come from.
3. **Pass the loader to `parse_ruleset()`** as `imports_loader=`. The imports are resolved while the text is parsed, and the result is a resolved rule set: the imported rules and `DATA` are merged into it, and `ruleset.is_resolved` is `True`. A document is read **once**, even if several rule sets import it or the imports form a cycle.

The example below first parses the rule set without a loader, to show what happens, and then with one.

Each rule in the example has an id, written after `RULE`, as in `RULE ex:directExposure { ... } WHERE { ... }`. An id is how a rule is named, for example in a plan. It is optional.

Here a base library defines the exposure rule, a second library builds on it, and the rule set imports the second one:

In [14]:
# Step 1: the libraries, as SRL text
base_library = """
PREFIX ex: <http://example.org/>
RULE ex:directExposure { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v }
"""

exposure_library = """
PREFIX ex: <http://example.org/>
IMPORTS <http://example.org/lib/base.srl>
RULE ex:inheritedExposure { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v }
"""

libraries = {
    "http://example.org/lib/base.srl": base_library,
    "http://example.org/lib/exposure.srl": exposure_library,
}


# Step 2: the loader - given the IRI of an import, returns that library's text
def library_loader(iri):
    return libraries[iri]


# The rule set to run: it imports the exposure library and adds one rule of its own
rules_with_imports = """
PREFIX ex: <http://example.org/>
IMPORTS <http://example.org/lib/exposure.srl>
RULE ex:markCritical { ?x ex:status ex:criticallyExposed } WHERE { ?x ex:exposedTo ?v }
"""

### Before: parsing without a loader

Without a loader, the text is parsed with its `IMPORTS` unresolved, and `ruleset.is_resolved` is `False`. StarLayer can then check the order of the rules in the text only, because an import can add a dependency that makes the combined rules impossible to order. `validate_sparql_tree()` reports each import left to resolve as a warning, which does not make the report fail, and the combined rule set is checked again when they are resolved. A rule set that is not resolved cannot be run: `infer()` raises an error.

In [15]:
from starlayer.errors import ExecutionError

# Parse without a loader: the imports stay unresolved
unresolved = srl.parse_ruleset(rules_with_imports)
print("is_resolved:", unresolved.is_resolved)

# Running it fails, because the imports are not resolved
try:
    unresolved.infer(base)
except ExecutionError as error:
    print(error)

is_resolved: False
SRL: this rule set has IMPORTS that are not resolved (<http://example.org/lib/exposure.srl>). Pass imports_loader= to infer(), query() or plan() to resolve them for the call, or resolve them once with srl.resolve_imports() or parse_ruleset(imports_loader=). A rule set with IMPORTS is never run with its imports ignored (§4.5).


### After: parsing with a loader

With `imports_loader=`, the imports are resolved while the text is parsed. The rule set has the three rules (the one written here and the two imported) and `is_resolved` is `True`, so there is no warning, and it runs without a loader:

In [16]:
# Parse with the loader: the imports are resolved while the text is parsed
with_imports = srl.parse_ruleset(rules_with_imports, imports_loader=library_loader)
print("is_resolved:", with_imports.is_resolved)
print("rules:", len(with_imports.rules))

# It runs without a loader
imported = with_imports.infer(base)
imported.bind("ex", EX)
show(imported)

# The imported rules ran: ex:frontend is exposed through ex:inheritedExposure and ex:directExposure
assert (ex("frontend"), ex("status"), ex("criticallyExposed")) in imported

is_resolved: True
rules: 3
@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:db ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .



### A loader for one call

A loader passed to `infer()`, `query()` or `plan()` resolves the imports for that call only. The rule set itself stays unresolved, and the loader is called again on the next call:

In [17]:
# Resolve the imports for this one call
per_call = unresolved.infer(base, imports_loader=library_loader)
print("triples derived:", len(per_call))

# The rule set itself is still unresolved
print("is_resolved:", unresolved.is_resolved)

# The same triples as the resolved rule set gave
assert set(per_call) == set(imported)

triples derived: 6
is_resolved: False


To see what the imports contributed, `srl.resolve_imports()` returns the rule set with the imported rules merged in and no imports left. It is what `parse_ruleset(..., imports_loader=)` does when it parses, applied to a rule set already in hand. Resolving the imports once and running the result many times avoids reading the libraries on every `infer()`:

In [18]:
resolved = srl.resolve_imports(unresolved, imports_loader=library_loader)
print(f"{len(unresolved.rules)} rule written, {len(resolved.rules)} after resolving the imports; imports left: {resolved.imports}")
print("is_resolved before:", unresolved.is_resolved, "| after:", resolved.is_resolved)

# the resolved rule set needs no loader
assert len(resolved.infer(base)) == len(imported)
assert len(resolved.rules) == 3

1 rule written, 3 after resolving the imports; imports left: []
is_resolved before: False | after: True


`plan()` shows how the rules will run once the imports are resolved. Each rule here has an id so the plan can name it. The plan lists the two imported rules, `ex:directExposure` and `ex:inheritedExposure`, together with the rule written in this rule set, `ex:markCritical`, and the edges show which depends on which.

All three share one stratum because none of them needs a later one: none uses `NOT`, and none is a run-once rule. A rule is only placed after the rules it reads when it reads them through `NOT` or when it is a run-once rule (section 04.02.4). A resolved rule set needs no loader, and `plan(imports_loader=...)` does the same for one that is not resolved.

In [19]:
plan = with_imports.plan()

names = {r.index: base.qname(r.id) for stratum in plan.strata for r in stratum.general + stratum.run_once}

for stratum in plan.strata:
    print(f"stratum {stratum.number}:", ", ".join(names[r.index] for r in stratum.general))
for edge in sorted(plan.edges, key=lambda e: (e.source, e.target)):
    print(f"{names[edge.source]} depends on {names[edge.target]} ({edge.label})")

assert set(names.values()) == {"ex:markCritical", "ex:directExposure", "ex:inheritedExposure"}

stratum 0: ex:markCritical, ex:inheritedExposure, ex:directExposure
ex:markCritical depends on ex:inheritedExposure (open)
ex:markCritical depends on ex:directExposure (open)
ex:inheritedExposure depends on ex:inheritedExposure (open)
ex:inheritedExposure depends on ex:directExposure (open)
